In [ ]:
!pip -q install trimesh networkx torch-geometric

In [ ]:
import trimesh
import torch
import numpy as np

from collections import defaultdict
from torch_geometric.data import Data


In [ ]:
stl_file = "Pivot Support.stl"

mesh = trimesh.load(stl_file)

In [ ]:
#for basic STL information

print(f"Vertices: {len(mesh.vertices)}")
print(f"Faces: {len(mesh.faces)}")
print(f"Edges: {len(mesh.edges)}")
print(f"Area: {mesh.area}")
print(f"Volume: {mesh.volume}")

Vertices: 330
Faces: 672
Edges: 2016
Area: 41975.67359937117
Volume: 227442.18357879692


In [ ]:
#Build edge into triangle map

edge_to_triangles = defaultdict(list)

for triangle_id, triangle in enumerate(mesh.faces):

  v0, v1, v2 = triangle

  edge0 = tuple(sorted([v0, v1]))
  edge1 = tuple(sorted([v1, v2]))
  edge2 = tuple(sorted([v2, v0]))

  for edge in [edge0, edge1, edge2]:
    edge_to_triangles[edge].append(triangle_id)

In [ ]:
#for the AAG

adjacency = set()

for edge, triangle_ids in edge_to_triangles.items():

    if len(triangle_ids) < 2:
        continue

    for i in range(len(triangle_ids)):

        for j in range(i + 1, len(triangle_ids)):

            triangle_a = triangle_ids[i]
            triangle_b = triangle_ids[j]

            adjacency.add(
                (triangle_a, triangle_b)
            )

In [ ]:
edge_index = []

for triangle_a, triangle_b in adjacency:

    edge_index.append(
        [triangle_a, triangle_b]
    )

    edge_index.append(
        [triangle_b, triangle_a]
    )


edge_index = torch.tensor(
    edge_index,
    dtype=torch.long
).t().contiguous()

In [ ]:
areas = mesh.area_faces

centroids = mesh.triangles_center

normals = mesh.face_normals


x = np.column_stack([
    areas,
    centroids,
    normals
])


x = torch.tensor(
    x,
    dtype=torch.float
)


In [ ]:
data = Data(
    x=x,
    edge_index=edge_index
)

In [ ]:
print()
print(
    "Number of graph nodes:",
    data.num_nodes
)

print(
    "Number of directed edges:",
    data.num_edges
)

print(
    "Number of undirected AAG connections:",
    len(adjacency)
)

print(
    "Number of node features:",
    data.num_node_features
)

print(
    "Node feature matrix:",
    data.x.shape
)

print(
    "Edge index matrix:",
    data.edge_index.shape
)


Number of graph nodes: 672
Number of directed edges: 2016
Number of undirected AAG connections: 1008
Number of node features: 7
Node feature matrix: torch.Size([672, 7])
Edge index matrix: torch.Size([2, 2016])


In [ ]:
#for visualization of the STL model
import plotly.graph_objects as go

fig = go.Figure(
    data=[
        go.Mesh3d(
            x=mesh.vertices[:, 0],
            y=mesh.vertices[:, 1],
            z=mesh.vertices[:, 2],
            i=mesh.faces[:, 0],
            j=mesh.faces[:, 1],
            k=mesh.faces[:, 2],
            intensity=mesh.face_normals[:, 2],
            colorscale="Viridis",
            showscale=True,
            colorbar=dict(
                title="Surface<br>Normal Z"
            ),
            flatshading=True
        )
    ]
)

fig.update_layout(
    title="STL / Triangle Mesh Visualization",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
        aspectmode="data"
    ),
    width=900,
    height=700
)

fig.show()

In [ ]:

#Visulize Triangel Centers


centroids = mesh.triangles_center

fig = go.Figure()

fig.add_trace(
    go.Mesh3d(
        x=mesh.vertices[:, 0],
        y=mesh.vertices[:, 1],
        z=mesh.vertices[:, 2],
        i=mesh.faces[:, 0],
        j=mesh.faces[:, 1],
        k=mesh.faces[:, 2],
        opacity=0.35,
        flatshading=True,
        showscale=False
    )
)

fig.add_trace(
    go.Scatter3d(
        x=centroids[:, 0],
        y=centroids[:, 1],
        z=centroids[:, 2],
        mode="markers",
        marker=dict(
            size=3,
            color="red"
        ),
        name="Graph Nodes"
    )
)

fig.update_layout(
    title="STL Triangles and Graph Nodes",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
        aspectmode="data"
    ),
    width=900,
    height=700
)

fig.show()

In [ ]:

# Visualize AAG on the STL file


fig = go.Figure()

# STL MODEL


fig.add_trace(
    go.Mesh3d(
        x=mesh.vertices[:, 0],
        y=mesh.vertices[:, 1],
        z=mesh.vertices[:, 2],
        i=mesh.faces[:, 0],
        j=mesh.faces[:, 1],
        k=mesh.faces[:, 2],
        opacity=0.25,
        flatshading=True,
        showscale=False,
        name="STL"
    )
)



# for graph nodes

fig.add_trace(
    go.Scatter3d(
        x=centroids[:, 0],
        y=centroids[:, 1],
        z=centroids[:, 2],
        mode="markers",
        marker=dict(
            size=3,
            color="red"
        ),
        name="AAG Nodes"
    )
)


# For graph edges
for triangle_a, triangle_b in adjacency:

    point_a = centroids[triangle_a]
    point_b = centroids[triangle_b]

    fig.add_trace(
        go.Scatter3d(
            x=[
                point_a[0],
                point_b[0]
            ],
            y=[
                point_a[1],
                point_b[1]
            ],
            z=[
                point_a[2],
                point_b[2]
            ],
            mode="lines",
            line=dict(
                width=1
            ),
            showlegend=False
        )
    )



# Layout


fig.update_layout(
    title="Attributed Adjacency Graph (AAG) on STL",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
        aspectmode="data"
    ),
    width=1000,
    height=800
)

fig.show()

In [ ]:

#AAG  graph only
fig = go.Figure()

# GRAPH EDGES

for triangle_a, triangle_b in adjacency:

    point_a = centroids[triangle_a]
    point_b = centroids[triangle_b]

    fig.add_trace(
        go.Scatter3d(
            x=[
                point_a[0],
                point_b[0]
            ],
            y=[
                point_a[1],
                point_b[1]
            ],
            z=[
                point_a[2],
                point_b[2]
            ],
            mode="lines",
            line=dict(
                width=2
            ),
            showlegend=False
        )
    )


# GRAPH NODES

fig.add_trace(
    go.Scatter3d(
        x=centroids[:, 0],
        y=centroids[:, 1],
        z=centroids[:, 2],
        mode="markers",
        marker=dict(
            size=5,
            color="red"
        ),
        name="Nodes"
    )
)


# LAYOUT

fig.update_layout(
    title="AAG — Graph Only",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
        aspectmode="data"
    ),
    width=1000,
    height=800
)

fig.show()